# Query analysis

Gold source rank per query, for every `(collection, alpha)` cell. One row per
query; six rank columns (two collections × three alphas). Low rank = found near
the top; `51` (= `top_k + 1`) = gold never retrieved within `top_k`.

In [10]:
from minerva_eval import load_all_results, gold_ranks, paired_chunk_deltas, paired_chunk_winloss
import pandas as pd

all_results = load_all_results()
print(f"{len(all_results)} results, {len({r.query_id for r in all_results})} queries")

330 results, 55 queries


## Gold rank table

Value is the gold's best source rank in that cell. Queries with multiple golds
collapse with `min` (success@1 — the best-placed gold). Rows are sorted hardest
first (largest worst-case rank on top), so trivially easy queries (rank 1 in
every cell) sink to the bottom. Green = low rank (easy), red = high rank (hard).

In [11]:
ranks = pd.DataFrame(vars(g) for g in gold_ranks(all_results))

table = ranks.pivot_table(
    index="query_id", columns=["collection", "alpha"],
    values="rank", aggfunc="min",          # multi-gold -> best-placed gold (success@1)
)
table = table.loc[table.max(axis=1).sort_values(ascending=False).index]

table.style.background_gradient(cmap="RdYlGn_r", axis=None)

## Paired chunk comparison — does contextualization help?

For each `(query, gold, alpha)`, pair the gold source's chunks between the two
collections on `chunk_index` (recovered by inverting the deterministic
`chunk_id` hash, since `chunk_id = sha256("{collection}:{source_id}:{index}")`
carries no content). Two metrics side by side per alpha:

- **avg**: mean signed rank change `baseline_rank − treatment_rank`. Positive =
  the treatment ranks the gold's chunks higher on average.
- **winloss**: `treatment_wins − baseline_wins`, i.e. how many chunks the
  treatment ranks better minus how many the baseline ranks better (ties ignored).

Each metric is colored on its own symmetric scale (their magnitudes differ).
Green = contextualization helps, red = it hurts.

In [12]:
BASELINE, TREATMENT = "wikipedia-nollm", "wikipedia-qwen2-5"

deltas = pd.DataFrame(vars(d) for d in paired_chunk_deltas(all_results, BASELINE, TREATMENT))
winloss = pd.DataFrame(vars(w) for w in paired_chunk_winloss(all_results, BASELINE, TREATMENT))

avg = deltas.pivot_table(index=["query_id", "gold"], columns="alpha", values="mean_delta")
wl = winloss.pivot_table(index=["query_id", "gold"], columns="alpha", values="net_wins")

# one block per alpha: avg column then winloss column
table = pd.concat({"avg": avg, "win-loss": wl}, axis=1)   # columns: (metric, alpha)
table.columns = table.columns.reorder_levels([1, 0])     # -> (alpha, metric)
table = table.sort_index(axis=1)                          # alpha asc, then avg before winloss
table = table.loc[table.abs().max(axis=1).sort_values(ascending=False).index]

# diverging scale centred on 0, per metric (avg and winloss have different magnitudes)
styled = table.style
for metric in ["avg", "win-loss"]:
    cols = [c for c in table.columns if c[1] == metric]
    m = table[cols].abs().max().max()
    styled = styled.background_gradient(cmap="RdYlGn", axis=None, vmin=-m, vmax=m, subset=cols)
styled